In [ ]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_free_agents, get_current_matchup
from fantasy.nba_stats import get_player_stats, get_games_this_week
from fantasy.analysis import project_team_categories, project_player, STAT_COLS
from fantasy.llm import build_waiver_prompt, ask_gemini
import pandas as pd

In [ ]:
matchup = get_current_matchup()
week_start, week_end = matchup["week_start"], matchup["week_end"]
my_roster = get_my_roster()

my_players = []
for p in my_roster:
    stats = get_player_stats(p["name"])
    games = get_games_this_week(p["team_abbr"], week_start, week_end)
    my_players.append({**p, "stats": stats, "games_remaining": games})

my_totals = project_team_categories(my_players)
# Identify bottom 3 categories by absolute projected value (normalized)
sorted_cats = sorted(my_totals, key=lambda c: my_totals[c])
weak_cats = sorted_cats[:3]
print(f"Weakest categories: {weak_cats}")

In [ ]:
fas = get_free_agents(count=50)

scored = []
for fa in fas:
    stats = get_player_stats(fa["name"])
    games = get_games_this_week(fa["team_abbr"], week_start, week_end)
    if stats is None:
        continue
    proj = project_player(stats, games, fa.get("status", ""))
    # Fit score: sum of projected values in my weak categories, boosted by games
    fit = sum(proj.get(c, 0) for c in weak_cats)
    scored.append({**fa, "games_remaining": games, "projected": proj, "fit_score": fit})

scored.sort(key=lambda x: x["fit_score"], reverse=True)
print(pd.DataFrame([{
    "Player": p["name"], "Pos": p["position"], "Status": p["status"] or "Active",
    "Games": p["games_remaining"], "Fit Score": round(p["fit_score"], 2)
} for p in scored[:20]]).to_string(index=False))

In [ ]:
prompt = build_waiver_prompt(scored, weak_cats)
advice = ask_gemini(prompt)
print("\n=== WAIVER WIRE PICKUPS ===\n")
print(advice)